# D3LM: A Discrete DNA Diffusion Language Model
**ArXivist-generated reproduction notebook**
Paper: https://arxiv.org/abs/2603.01780 (Yang, Liu, Cao, Su; MLGenX 2026)
Generated: 2026-07-25

This notebook (1) verifies the D3LM **masked-diffusion algorithm** on CPU (no weights),
(2) loads the **official D3LM-R weights** (HuggingFace), and (3) **generates** DNA and
evaluates it with real metrics (GC ratio, diversity, novelty, motif correlation). Use a
**GPU runtime** for fast 2048bp generation.

## 1. Upload the repo zip

In [ ]:
from google.colab import files
files.upload()                       # pick arxiv_2603_001780.zip
!unzip -oq arxiv_2603_001780.zip
%cd arxiv_2603_001780
import os; print('cwd:', os.getcwd(), '| has src/d3lm:', os.path.isdir('src/d3lm'))

## 2. Environment + install

In [ ]:
import sys, torch, subprocess
print(f"Python {sys.version.split()[0]} | PyTorch {torch.__version__} | CUDA {torch.cuda.is_available()}")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
r = subprocess.run(["pip","install","-q","-e","."], capture_output=True, text=True)
print((r.stdout or r.stderr)[-400:])

## 3. Verify the masked-diffusion algorithm (CPU, no weights) — 8 tests

Checks the paper's core: 6-mer vocab, forward masking (~t), the **1/t-weighted CE loss**
(Eq 2, exact 1/t scaling), and **iterative unmasking** (Eq 4, oracle recovery).

In [ ]:
open("conftest.py","a").close()
import subprocess
r = subprocess.run(["python","-m","pytest","tests/","-q"], capture_output=True, text=True)
print(r.stdout[-600:]); print(r.stderr[-200:] if r.returncode else "all tests passed")

## 4. Paper overview

**Problem.** DNA needs bidirectional context (enhancers act up- AND down-stream) *and*
generation. BERT-style = bidirectional but non-generative; autoregressive = generative but causal.

**D3LM.** Masked diffusion on the NT-v2 backbone:
- forward: mask each token w.p. `t ~ U[0,1]`
- loss (Eq 2): `L = -E[ (1/t) Σ 1[xt=M] log p(x0|xt) ]`
- generate (Eq 4): all-`[M]` → T=50 steps of predict + unmask (random order, τ=1.1)

D3LM-R hits **SFID 10.92** (Truth 7.85; HyenaDNA 29.16; DiscDiff 62.74).

## 5. Load the official D3LM-R weights

`Hengchang-Liu/D3LM-scratch` via AutoModelForMaskedLM(trust_remote_code=True).

In [ ]:
!python data/download.py --weights D3LM-R
from src.d3lm.models.d3lm import D3LMGenerator
gen = D3LMGenerator.from_pretrained("D3LM-R", device=str(device))
print("loaded D3LM-R")

## 6. Generate DNA (masked diffusion)

Start small (a few 256bp sequences) to confirm generation runs, then scale up.

In [ ]:
seqs = gen.generate(n=8, length=256, steps=50, temperature=1.1)
for s in seqs[:2]:
    print(len(s), "bp:", s[:80], "...")
# save a batch
import os; os.makedirs("results", exist_ok=True)
with open("results/gen.fasta","w") as fh:
    for i,s in enumerate(seqs): fh.write(f">d3lm_{i}\n{s}\n")
print("wrote", len(seqs), "sequences")

## 7. Evaluate (real metrics — no Sei needed)

GC ratio (Chargaff parity ~1.0), diversity, novelty, motif correlation. Paper D3LM-R
GC ratio = 1.07 (Truth 1.06).

In [ ]:
from src.d3lm.evaluation.metrics import gc_ratio, diversity, novelty
from src.d3lm.evaluation.motif import all_motif_correlations
from src.d3lm.data.epd_gendna import load_sequences
ref = load_sequences("epd_gendna","test","data/",length=256)
print("GC ratio:", round(gc_ratio(seqs),4), "(paper D3LM-R 1.07, Truth 1.06)")
print("Diversity:", round(diversity(seqs),2))
print("Novelty:", round(novelty(seqs, ref),2))
print("Motif corr:", {k:round(v,3) for k,v in all_motif_correlations(seqs, ref).items()})

## 8. Paper results for comparison (Table 1, 2048bp)

In [ ]:
paper = {
  "Truth (real DNA) SFID": 7.85, "D3LM-R SFID": 10.92, "D3LM SFID": 25.21,
  "HyenaDNA SFID": 29.16, "DiscDiff SFID": 62.74, "Evo SFID": 1359.98,
  "D3LM-R GC ratio": 1.07, "Truth GC ratio": 1.06,
}
for k,v in paper.items(): print(f"  {k:26s} {v}")
print("\nNote: full SFID needs the Sei genomic CNN. GC ratio + diversity + motif")
print("correlation are cheap REAL metrics that already separate good vs collapsed models.")
print("Paste your GC ratio / metrics back to ArXivist for the Stage 6 comparison.")

## What to do next

- Scale generation to 1000 x 2048bp: `!python generate.py --config configs/config.yaml`
- Try D3LM (understanding variant): `--weights D3LM`
- For true SFID, supply a Sei embedder to `evaluation/metrics.py:sfid`.
- Paste your metrics back to ArXivist (Stage 6).

**Recipe (paper):** masked diffusion, T=50 steps, temperature 1.1, random unmask order, 6-mer tokens.